In [1]:
"""
PINN quantique 3D cinematic avec PyVista.

But est : 

Créer une vraie animation 3D stylée d'un PINN qui apprend une fonction d'onde
solution de l'équation de Schrödinger libre :

    i ∂ψ/∂t = - 1/2 ∂²ψ/∂x²

On écrit :

    ψ(x,t) = u(x,t) + i v(x,t)

Le réseau de neurones prend (x,t) en entrée et prédit :

    (u,v) = (Re ψ, Im ψ)

Installation :

pip install torch numpy pyvista imageio imageio-ffmpeg

Sur Mac, si l'export MP4 pose problème :
brew install ffmpeg

Exécution : 

python pinn_schrodinger_quantum_wave_animation.py

Sorties :

    pinn_quantum_wave_cinematic.mp4

"""

import os
import numpy as np
import torch
import torch.nn as nn
import pyvista as pv


# 1. Configuration générale
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(7)
np.random.seed(7)

# Domaine physique
X_MIN, X_MAX = -np.pi, np.pi
T_MIN, T_MAX = 0.0, 1.0

# Superposition de deux modes quantiques.
# C'est plus joli qu'une simple onde plane car |ψ|² présente des interférences.
K1 = 1.0
K2 = 2.0
A1 = 0.75
A2 = 0.45

# Temps fixé pour visualiser la courbe complexe x -> (x, Re ψ, Im ψ)
T_VIEW = 0.75

# Hyperparamètres PINN
N_IC = 256
N_BC = 128
N_F = 4096
EPOCHS = 6000
SNAPSHOT_EVERY = 150
LEARNING_RATE = 1e-3

# Poids des termes de loss
LAMBDA_IC = 20.0
LAMBDA_BC = 5.0
LAMBDA_PDE = 1.0

# Grilles de rendu
NX_SURF = 100
NT_SURF = 65
NX_CURVE = 400

# Rendu vidéo
OUTPUT_MP4 = "pinn_quantum_wave_cinematic.mp4"
FPS = 24
CAMERA_ORBIT_DEGREES = 330

# Si PyVista bug en environnement sans fenêtre graphique, garder cette ligne.
pv.OFF_SCREEN = True


# 2. Solution analytique

def psi_exact_np(x, t):
    """
    Solution exacte de Schrödinger libre.

    Pour un mode :
        ψ_k(x,t) = exp(i(kx - ωt))

    avec :
        ω = k² / 2

    Ici on prend une superposition :
        ψ = A1 exp(i(k1 x - ω1 t)) + A2 exp(i(k2 x - ω2 t))
    """
    omega1 = K1**2 / 2.0
    omega2 = K2**2 / 2.0
    return (
        A1 * np.exp(1j * (K1 * x - omega1 * t))
        + A2 * np.exp(1j * (K2 * x - omega2 * t))
    )


def exact_torch(x, t):
    """Version PyTorch de la solution exacte."""
    omega1 = K1**2 / 2.0
    omega2 = K2**2 / 2.0

    phase1 = K1 * x - omega1 * t
    phase2 = K2 * x - omega2 * t

    u = A1 * torch.cos(phase1) + A2 * torch.cos(phase2)
    v = A1 * torch.sin(phase1) + A2 * torch.sin(phase2)
    return u, v


# 3. Réseau de neurones PINN

class PINN(nn.Module):
    def __init__(self, width=80, depth=5):
        super().__init__()

        layers = [nn.Linear(2, width), nn.Tanh()]

        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]

        layers.append(nn.Linear(width, 2))
        self.net = nn.Sequential(*layers)
        self.init_weights()

    def init_weights(self):
        for module in self.net:
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


# 4. Résidu de Schrödinger

def gradients(y, x):
    return torch.autograd.grad(
        y,
        x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]


def schrodinger_residual(model, x, t):
    """
    Schrödinger libre :
        i ψ_t = -1/2 ψ_xx

    Pour ψ = u + iv :
        u_t + 1/2 v_xx = 0
        v_t - 1/2 u_xx = 0
    """
    pred = model(x, t)
    u = pred[:, 0:1]
    v = pred[:, 1:2]

    u_t = gradients(u, t)
    v_t = gradients(v, t)

    u_x = gradients(u, x)
    v_x = gradients(v, x)

    u_xx = gradients(u_x, x)
    v_xx = gradients(v_x, x)

    r_u = u_t + 0.5 * v_xx
    r_v = v_t - 0.5 * u_xx

    return r_u, r_v


def to_tensor(array, requires_grad=False):
    return torch.tensor(
        array,
        dtype=torch.float32,
        device=DEVICE,
        requires_grad=requires_grad,
    )


# 5. Données d'entraînement

# Condition initiale : t = 0
x_ic_np = np.random.uniform(X_MIN, X_MAX, (N_IC, 1))
t_ic_np = np.zeros_like(x_ic_np)

x_ic = to_tensor(x_ic_np)
t_ic = to_tensor(t_ic_np)
u_ic, v_ic = exact_torch(x_ic, t_ic)

# Conditions aux bords : x = -pi et x = pi
# Les modes k=1 et k=2 sont compatibles avec la périodicité.
t_bc_np = np.random.uniform(T_MIN, T_MAX, (N_BC, 1))
x_left_np = X_MIN * np.ones_like(t_bc_np)
x_right_np = X_MAX * np.ones_like(t_bc_np)

x_left = to_tensor(x_left_np)
x_right = to_tensor(x_right_np)
t_bc = to_tensor(t_bc_np)

u_left, v_left = exact_torch(x_left, t_bc)
u_right, v_right = exact_torch(x_right, t_bc)

# Points de collocation pour imposer l'équation physique
x_f_np = np.random.uniform(X_MIN, X_MAX, (N_F, 1))
t_f_np = np.random.uniform(T_MIN, T_MAX, (N_F, 1))

x_f = to_tensor(x_f_np, requires_grad=True)
t_f = to_tensor(t_f_np, requires_grad=True)


# 6. Grilles pour snapshots

x_surf_np = np.linspace(X_MIN, X_MAX, NX_SURF)
t_surf_np = np.linspace(T_MIN, T_MAX, NT_SURF)
X_surf_np, T_surf_np = np.meshgrid(x_surf_np, t_surf_np)

x_surf_flat = X_surf_np.reshape(-1, 1)
t_surf_flat = T_surf_np.reshape(-1, 1)

x_surf = to_tensor(x_surf_flat)
t_surf = to_tensor(t_surf_flat)

psi_surf_true = psi_exact_np(X_surf_np, T_surf_np)
prob_surf_true = np.abs(psi_surf_true) ** 2

x_curve_np = np.linspace(X_MIN, X_MAX, NX_CURVE).reshape(-1, 1)
t_curve_np = T_VIEW * np.ones_like(x_curve_np)

x_curve = to_tensor(x_curve_np)
t_curve = to_tensor(t_curve_np)

psi_curve_true = psi_exact_np(x_curve_np.flatten(), T_VIEW)
u_curve_true = np.real(psi_curve_true)
v_curve_true = np.imag(psi_curve_true)


# 7. Entraînement

model = PINN(width=80, depth=5).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
mse = nn.MSELoss()

snapshots = []
loss_history = []

print(f"Device utilisé : {DEVICE}")
print("Entraînement du PINN quantique...")

for epoch in range(EPOCHS + 1):
    optimizer.zero_grad()

    # Condition initiale
    pred_ic = model(x_ic, t_ic)
    loss_ic = mse(pred_ic[:, 0:1], u_ic) + mse(pred_ic[:, 1:2], v_ic)

    # Conditions aux bords
    pred_left = model(x_left, t_bc)
    pred_right = model(x_right, t_bc)

    loss_bc = (
        mse(pred_left[:, 0:1], u_left)
        + mse(pred_left[:, 1:2], v_left)
        + mse(pred_right[:, 0:1], u_right)
        + mse(pred_right[:, 1:2], v_right)
    )

    # Résidu physique
    r_u, r_v = schrodinger_residual(model, x_f, t_f)
    loss_pde = mse(r_u, torch.zeros_like(r_u)) + mse(r_v, torch.zeros_like(r_v))

    loss = LAMBDA_IC * loss_ic + LAMBDA_BC * loss_bc + LAMBDA_PDE * loss_pde
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % SNAPSHOT_EVERY == 0:
        with torch.no_grad():
            pred_surf = model(x_surf, t_surf).detach().cpu().numpy()
            u_surf = pred_surf[:, 0].reshape(NT_SURF, NX_SURF)
            v_surf = pred_surf[:, 1].reshape(NT_SURF, NX_SURF)
            prob_surf_pred = u_surf**2 + v_surf**2

            pred_curve = model(x_curve, t_curve).detach().cpu().numpy()
            u_curve_pred = pred_curve[:, 0]
            v_curve_pred = pred_curve[:, 1]

            snapshots.append(
                {
                    "epoch": epoch,
                    "loss": loss.item(),
                    "loss_ic": loss_ic.item(),
                    "loss_bc": loss_bc.item(),
                    "loss_pde": loss_pde.item(),
                    "prob_surface": prob_surf_pred,
                    "u_curve": u_curve_pred,
                    "v_curve": v_curve_pred,
                }
            )

        print(
            f"epoch {epoch:5d} | "
            f"loss={loss.item():.3e} | "
            f"IC={loss_ic.item():.3e} | "
            f"BC={loss_bc.item():.3e} | "
            f"PDE={loss_pde.item():.3e}"
        )


# 8. Fonctions PyVista pour créer les objets 3D

def make_surface_grid(prob):
    """
    Construit une surface PyVista à partir de z = |ψ|².

    On décale et scale les axes pour que la scène soit plus lisible :
        x : position
        y : temps, amplifié
        z : densité |ψ|²
    """
    X = X_surf_np
    Y = 4.0 * T_surf_np - 2.0
    Z = 1.25 * prob

    grid = pv.StructuredGrid(X, Y, Z)
    grid["density"] = prob.ravel(order="F")
    return grid


def make_complex_curve(u, v):
    """
    Courbe 3D de la fonction d'onde complexe :
        x -> (x, Re ψ, Im ψ)

    On la décale vers la droite de la scène pour ne pas se mélanger avec la surface.
    """
    x = x_curve_np.flatten() + 8.0
    y = 1.2 * u
    z = 1.2 * v + 1.0

    points = np.column_stack([x, y, z])
    curve = pv.PolyData(points)

    lines = np.empty((NX_CURVE - 1, 3), dtype=np.int64)
    lines[:, 0] = 2
    lines[:, 1] = np.arange(0, NX_CURVE - 1)
    lines[:, 2] = np.arange(1, NX_CURVE)
    curve.lines = lines

    return curve.tube(radius=0.035, n_sides=16)


def make_neural_network_actor(plotter, progress=0.0):
    """
    Ajoute un petit réseau de neurones stylisé à gauche.
    Ce n'est pas le vrai graphe interne de PyTorch : c'est une représentation visuelle.
    """
    layer_sizes = [2, 5, 5, 2]
    layer_x = np.linspace(-8.8, -5.2, len(layer_sizes))
    nodes = []

    for i, n in enumerate(layer_sizes):
        ys = np.linspace(-1.4, 1.4, n)
        for y in ys:
            nodes.append((layer_x[i], y, 1.4))
            sphere = pv.Sphere(radius=0.09 + 0.03 * progress, center=(layer_x[i], y, 1.4))
            plotter.add_mesh(
                sphere,
                color=(0.3 + 0.4 * progress, 0.8, 1.0),
                emissive=True,
                smooth_shading=True,
            )

    # Connexions
    layer_nodes = []
    cursor = 0
    for n in layer_sizes:
        layer_nodes.append(nodes[cursor: cursor + n])
        cursor += n

    for i in range(len(layer_nodes) - 1):
        for p1 in layer_nodes[i]:
            for p2 in layer_nodes[i + 1]:
                line = pv.Line(p1, p2)
                tube = line.tube(radius=0.008 + 0.006 * progress)
                plotter.add_mesh(
                    tube,
                    color=(0.1, 0.45 + 0.35 * progress, 1.0),
                    opacity=0.22 + 0.35 * progress,
                    emissive=True,
                )


# 9. Rendu cinematic PyVista

print("Préparation du rendu PyVista cinematic...")

# Premier snapshot
first = snapshots[0]

# Grilles initiales
surface_mesh = make_surface_grid(first["prob_surface"])
exact_surface_mesh = make_surface_grid(prob_surf_true)
complex_curve_mesh = make_complex_curve(first["u_curve"], first["v_curve"])
exact_curve_mesh = make_complex_curve(u_curve_true, v_curve_true)

# Plotter
plotter = pv.Plotter(
    off_screen=True,
    window_size=(1920, 1080),
)
plotter.set_background("#02030a")

# Lumières
plotter.remove_all_lights()
key_light = pv.Light(position=(0, -7, 8), focal_point=(0, 0, 0), color="white", intensity=1.1)
rim_light = pv.Light(position=(9, 6, 5), focal_point=(0, 0, 0), color="#55ccff", intensity=0.9)
low_light = pv.Light(position=(-8, 3, 2), focal_point=(0, 0, 0), color="#aa55ff", intensity=0.45)
plotter.add_light(key_light)
plotter.add_light(rim_light)
plotter.add_light(low_light)

# Surface exacte : wireframe blanc transparent
plotter.add_mesh(
    exact_surface_mesh,
    style="wireframe",
    color="white",
    opacity=0.22,
    line_width=1,
)

# Surface prédite : surface lumineuse colorée
surface_actor = plotter.add_mesh(
    surface_mesh,
    scalars="density",
    cmap="plasma",
    clim=[0.0, float(np.max(prob_surf_true))],
    smooth_shading=True,
    specular=0.65,
    metallic=0.15,
    roughness=0.28,
)

# Courbe complexe exacte
plotter.add_mesh(
    exact_curve_mesh,
    color="white",
    opacity=0.28,
    smooth_shading=True,
)

# Courbe complexe prédite
curve_actor = plotter.add_mesh(
    complex_curve_mesh,
    color="#00e5ff",
    smooth_shading=True,
    specular=0.8,
    emissive=True,
)

# Axes / grille discrète
plotter.show_grid(
    color="white",
    grid="back",
    location="outer",
    xlabel="x",
    ylabel="t / Re ψ",
    zlabel="|ψ|² / Im ψ",
    font_size=10,
)

# Textes
plotter.add_text(
    "PINN quantique — apprentissage de ψ(x,t)",
    position="upper_left",
    font_size=20,
    color="white",
)
info_actor = plotter.add_text(
    "",
    position="lower_left",
    font_size=13,
    color="white",
)

# Réseau de neurones stylisé initial
make_neural_network_actor(plotter, progress=0.0)

# Position caméra initiale
plotter.camera_position = [
    (8.5, -10.5, 6.0),   # position caméra
    (1.0, 0.0, 0.9),     # point regardé
    (0.0, 0.0, 1.0),     # vecteur vertical
]
plotter.camera.zoom(1.05)

# Ouvrir le fichier vidéo
if os.path.exists(OUTPUT_MP4):
    os.remove(OUTPUT_MP4)

plotter.open_movie(OUTPUT_MP4, framerate=FPS, quality=9)

print(f"Écriture de la vidéo : {OUTPUT_MP4}")

# Nombre de frames supplémentaires pour rendre l'animation plus fluide.
# On interpole entre snapshots successifs.
INTERP_STEPS = 4

frames = []
for i in range(len(snapshots) - 1):
    s0 = snapshots[i]
    s1 = snapshots[i + 1]
    for j in range(INTERP_STEPS):
        alpha = j / INTERP_STEPS
        prob = (1.0 - alpha) * s0["prob_surface"] + alpha * s1["prob_surface"]
        u = (1.0 - alpha) * s0["u_curve"] + alpha * s1["u_curve"]
        v = (1.0 - alpha) * s0["v_curve"] + alpha * s1["v_curve"]
        loss = (1.0 - alpha) * s0["loss"] + alpha * s1["loss"]
        epoch = int((1.0 - alpha) * s0["epoch"] + alpha * s1["epoch"])
        frames.append((prob, u, v, loss, epoch))

# Ajouter le dernier snapshot
last = snapshots[-1]
frames.append((last["prob_surface"], last["u_curve"], last["v_curve"], last["loss"], last["epoch"]))

for frame_id, (prob, u, v, loss, epoch) in enumerate(frames):
    progress = frame_id / max(1, len(frames) - 1)

    # Mise à jour surface
    new_surface = make_surface_grid(prob)
    surface_actor.mapper.SetInputData(new_surface)

    # Mise à jour courbe complexe
    new_curve = make_complex_curve(u, v)
    curve_actor.mapper.SetInputData(new_curve)

    # Caméra orbitale
    theta = np.deg2rad(-55 + CAMERA_ORBIT_DEGREES * progress)
    radius = 13.5
    cam_x = 1.0 + radius * np.cos(theta)
    cam_y = radius * np.sin(theta)
    cam_z = 5.2 + 1.0 * np.sin(2.0 * np.pi * progress)

    plotter.camera_position = [
        (cam_x, cam_y, cam_z),
        (1.0, 0.0, 0.9),
        (0.0, 0.0, 1.0),
    ]

    # Texte dynamique
    info = (
        f"epoch = {epoch}    "
        f"loss = {loss:.2e}    "
        f"contrainte physique : i∂tψ = -1/2 ∂xxψ"
    )
    info_actor.SetText(2, info)

    plotter.write_frame()

plotter.close()

print("Terminé.")
print(f"Vidéo sauvegardée : {OUTPUT_MP4}")


Device utilisé : cpu
Entraînement du PINN quantique...
epoch     0 | loss=2.038e+01 | IC=7.178e-01 | BC=1.180e+00 | PDE=1.293e-01
epoch   150 | loss=5.030e-01 | IC=1.034e-02 | BC=3.536e-02 | PDE=1.194e-01
epoch   300 | loss=1.540e-01 | IC=1.741e-03 | BC=1.051e-02 | PDE=6.659e-02
epoch   450 | loss=8.226e-02 | IC=4.411e-04 | BC=5.398e-03 | PDE=4.645e-02
epoch   600 | loss=5.685e-02 | IC=2.558e-04 | BC=2.644e-03 | PDE=3.851e-02
epoch   750 | loss=4.557e-02 | IC=1.966e-04 | BC=1.767e-03 | PDE=3.281e-02
epoch   900 | loss=3.892e-02 | IC=1.593e-04 | BC=1.466e-03 | PDE=2.840e-02
epoch  1050 | loss=3.391e-02 | IC=1.311e-04 | BC=1.273e-03 | PDE=2.492e-02
epoch  1200 | loss=9.065e-02 | IC=1.598e-03 | BC=7.390e-03 | PDE=2.174e-02
epoch  1350 | loss=2.886e-02 | IC=2.090e-04 | BC=1.158e-03 | PDE=1.889e-02
epoch  1500 | loss=2.130e-02 | IC=6.734e-05 | BC=8.327e-04 | PDE=1.579e-02
epoch  1650 | loss=1.799e-02 | IC=5.310e-05 | BC=7.372e-04 | PDE=1.324e-02
epoch  1800 | loss=1.499e-02 | IC=5.494e-05 |

/var/folders/tv/n34_b53j7zj29rwp532_xsmw0000gn/T/ipykernel_54186/3996160189.py:511: PyVistaDeprecationWarning: `xlabel` is deprecated. Use `xtitle` instead.
  plotter.show_grid(
/var/folders/tv/n34_b53j7zj29rwp532_xsmw0000gn/T/ipykernel_54186/3996160189.py:511: PyVistaDeprecationWarning: `ylabel` is deprecated. Use `ytitle` instead.
  plotter.show_grid(
/var/folders/tv/n34_b53j7zj29rwp532_xsmw0000gn/T/ipykernel_54186/3996160189.py:511: PyVistaDeprecationWarning: `zlabel` is deprecated. Use `ztitle` instead.
  plotter.show_grid(


Écriture de la vidéo : pinn_quantum_wave_cinematic.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Terminé.
Vidéo sauvegardée : pinn_quantum_wave_cinematic.mp4
